# Constructed rolling 3-month IPCA forecast errors

This notebook uses the last Focus IPCA observation in each month from November 2001 through January 2025, the **same 279 forecast months** as the rolling 12-month notebook. It compounds 3 published *monthly medians* for the forecast month and the following 2 months, then compares that constructed forecast with realized IPCA over exactly those months. The realized rate uses the [IBGE monthly price index](https://sidra.ibge.gov.br/tabela/1737): `100 × (index at window end / index before window start − 1)`.

**Methodological limit:** Focus does not publish a direct rolling 3-month median in this dataset. Compounding medians calculated separately for each target month is a **Focus-based constructed forecast**, not the median of institution-level 3-month forecasts. Those individual forecasts are unavailable in this repository. Monthly Focus statistics come from the [BCB expectations dataset](https://dadosabertos.bcb.gov.br/dataset/expectativas-mercado).

## 1. Load the local Focus snapshot and IBGE index

The IBGE index CSV is generated locally using `projects/download_ibge_ipca.R`. The Focus SQLite snapshot also remains local.

In [ ]:
suppressPackageStartupMessages({
  library(DBI)
  library(RSQLite)
  library(jsonlite)
})

horizon_months <- 3L
project_root <- if (file.exists('dados/atual.json')) {
  '.'
} else if (file.exists('../dados/atual.json')) {
  '..'
} else {
  stop('Cannot find dados/atual.json; run from the repository root or projects/.')
}
snapshot <- fromJSON(file.path(project_root, 'dados/atual.json'))
ibge <- read.csv(file.path(project_root, 'projects/data/ibge_ipca_monthly_index_200110_202512.csv'),
                 colClasses = c('character', 'numeric'))
expected_index_months <- format(seq(as.Date('2001-10-01'), as.Date('2025-12-01'), by = 'month'), '%Y%m')
stopifnot(identical(ibge$month, expected_index_months),
          all(is.finite(ibge$ipca_index)), all(ibge$ipca_index > 0))
cat('Focus snapshot:', snapshot$coleta, '\n')
cat('Horizon:', horizon_months, 'months; IBGE index months:', nrow(ibge), '\n')

## 2. Construct a rolling 3-month forecast

At each month-end date, collect the monthly IPCA medians for the current month through month 2. The expected count is exactly 3 nonmissing monthly inputs at every anchor. Compound them, rather than taking an arithmetic average.

In [ ]:
con <- dbConnect(SQLite(), file.path(project_root, snapshot$banco))
monthly <- tryCatch(dbGetQuery(con, '
  WITH anchors AS (
    SELECT substr(Data, 1, 7) AS forecast_month, max(Data) AS forecast_date
    FROM ExpectativaMercadoMensais
    WHERE Indicador = ? AND baseCalculo = ? AND Data >= ? AND Data < ?
    GROUP BY substr(Data, 1, 7)
  ), aligned AS (
    SELECT a.forecast_month, a.forecast_date,
           m.DataReferencia AS target_month, m.Mediana AS monthly_forecast_pct,
           (CAST(substr(m.DataReferencia, 4, 4) AS INTEGER)
             - CAST(substr(a.forecast_month, 1, 4) AS INTEGER)) * 12
             + CAST(substr(m.DataReferencia, 1, 2) AS INTEGER)
             - CAST(substr(a.forecast_month, 6, 2) AS INTEGER) AS months_ahead
    FROM anchors a CROSS JOIN ExpectativaMercadoMensais m
    WHERE m.Data = a.forecast_date AND m.Indicador = ? AND m.baseCalculo = ?
  )
  SELECT forecast_month, forecast_date, target_month,
         monthly_forecast_pct, months_ahead
  FROM aligned WHERE months_ahead BETWEEN 0 AND ?
  ORDER BY forecast_month, months_ahead
', params = list('IPCA', 0L, '2001-11-01', '2025-02-01',
                'IPCA', 0L, horizon_months - 1L)),
finally = dbDisconnect(con))

parts <- split(monthly, monthly$forecast_month)
expected_forecast_months <- format(seq(as.Date('2001-11-01'), as.Date('2025-01-01'), by = 'month'), '%Y-%m')
stopifnot(identical(names(parts), expected_forecast_months),
          all(vapply(parts, nrow, integer(1)) == horizon_months))
focus <- do.call(rbind, lapply(parts, function(part) {
  if (anyNA(part$monthly_forecast_pct) ||
      !all(part$months_ahead == seq.int(0L, horizon_months - 1L))) {
    stop('Missing monthly forecast for ', part$forecast_month[1])
  }
  month <- part$forecast_month[1]
  first_day <- as.Date(paste0(month, '-01'))
  lag_days <- as.integer(seq(first_day, by = 'month', length.out = 2)[2] - 1
                         - as.Date(part$forecast_date[1]))
  if (lag_days < 0L || lag_days > 7L) stop('Stale month-end forecast for ', month)
  end_ref <- tail(part$target_month, 1)
  data.frame(forecast_month = month, forecast_date = part$forecast_date[1],
             horizon_months = horizon_months, monthly_inputs = nrow(part),
             days_before_month_end = lag_days,
             index_before_window = format(first_day - 1, '%Y%m'),
             outcome_end_month = paste0(substr(end_ref, 4, 7), substr(end_ref, 1, 2)),
             forecast_ipca_pct = 100 * (prod(1 + part$monthly_forecast_pct / 100) - 1))
}))
row.names(focus) <- NULL
stopifnot(nrow(focus) == 279L)
cat('Constructed monthly forecasts:', nrow(focus), '\n')
head(focus, 3)

## 3. Match each forecast to realized IPCA

The realized window starts with the forecast month and ends 2 months later. Use the IBGE index from the month before the window as the denominator. Signed error is constructed forecast minus realized inflation, in percentage points.

In [ ]:
start_row <- match(focus$index_before_window, ibge$month)
end_row <- match(focus$outcome_end_month, ibge$month)
stopifnot(!anyNA(start_row), !anyNA(end_row))
focus$realized_ipca_pct <- 100 * (ibge$ipca_index[end_row] / ibge$ipca_index[start_row] - 1)
focus$error_pp <- focus$forecast_ipca_pct - focus$realized_ipca_pct
focus$absolute_error_pp <- abs(focus$error_pp)
stopifnot(all(is.finite(focus$error_pp)))

selected <- focus[c('forecast_month', 'forecast_date', 'outcome_end_month',
                    'forecast_ipca_pct', 'realized_ipca_pct',
                    'error_pp', 'absolute_error_pp')]
print(head(selected, 6), row.names = FALSE, digits = 4)
print(tail(selected, 6), row.names = FALSE, digits = 4)

## 4. Summarize the errors

The 3-month realized windows overlap by 2 months. The 279 errors are therefore correlated, even though the forecasts are sampled only once per month.

In [ ]:
summary_rolling <- data.frame(
  months = nrow(focus),
  mean_signed_error_pp = mean(focus$error_pp),
  mean_absolute_error_pp = mean(focus$absolute_error_pp),
  median_absolute_error_pp = median(focus$absolute_error_pp),
  sd_error_pp = sd(focus$error_pp),
  rmse_pp = sqrt(mean(focus$error_pp^2))
)
print(summary_rolling, row.names = FALSE, digits = 4)

## 5. Forecast, realized IPCA, and error

The x-axis is the **forecast month**. The realized line shows a later outcome, which was unknown when the forecast was recorded.

In [ ]:
plot_month <- as.Date(paste0(focus$forecast_month, '-01'))
options(repr.plot.width = 10, repr.plot.height = 8)
old_par <- par(no.readonly = TRUE)
par(bg = 'white', mfrow = c(2, 1), mar = c(2, 4.5, 3, 1))

plot(plot_month, focus$forecast_ipca_pct, type = 'l',
     col = 'steelblue4', lwd = 1.5, xaxt = 'n', xlab = '',
     ylim = range(c(focus$forecast_ipca_pct, focus$realized_ipca_pct)),
     ylab = 'IPCA (%)', main = 'Focus-based forecast vs subsequent realized IPCA')
lines(plot_month, focus$realized_ipca_pct, col = 'darkorange2', lwd = 1.5)
legend('topright', c('Constructed 3-month forecast', 'Realized next 3 months'),
       col = c('steelblue4', 'darkorange2'), lty = 1, lwd = 1.5, bty = 'n')

par(mar = c(5, 4.5, 3, 1))
plot(plot_month, focus$error_pp, type = 'l', col = 'grey30',
     xlab = 'Forecast month', ylab = 'Error (p.p.)',
     main = 'Forecast minus realized IPCA')
abline(h = 0, col = 'red', lty = 2)
par(old_par)

Static preview from Focus snapshot `20261029T204913Z`:

![Rolling 3-month Focus-based forecast, realized IPCA, and error](figures/ipca_rolling_3m_forecast_realized_error.png)

## 6. Distribution of signed errors

Negative values indicate underprediction; positive values indicate overprediction. The blue solid line is the mean; dotted lines mark one sample standard deviation on either side of the mean.

In [ ]:
mean_error <- mean(focus$error_pp)
sd_error <- sd(focus$error_pp)

par(bg = 'white')
hist(focus$error_pp, breaks = 20, col = 'lightblue', border = 'white',
     main = 'Distribution of rolling 3-month forecast errors',
     xlab = 'Forecast minus realized IPCA (p.p.)')
abline(v = 0, col = 'red', lty = 2, lwd = 2)
abline(v = mean_error, col = 'navy', lwd = 2)
abline(v = mean_error + c(-sd_error, sd_error), col = 'navy', lty = 3, lwd = 2)
legend('topleft',
       legend = c(sprintf('Mean = %.2f p.p.', mean_error),
                  sprintf('SD = %.2f p.p. (mean +/− SD)', sd_error),
                  'Zero error'),
       col = c('navy', 'navy', 'red'), lty = c(1, 3, 2), lwd = 2,
       bty = 'n', cex = 0.85)

Static preview from the same snapshot:

![Rolling 3-month error histogram with mean and standard deviation](figures/ipca_rolling_3m_error_histogram.png)